In [57]:
from ultralytics import YOLO
import cv2
import numpy as np

In [58]:
model = YOLO("runs/detect/train2/weights/best.pt")

In [59]:
video_path = "../videos/umpire_view.mp4"

cap = cv2.VideoCapture(video_path)

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fps = int(cap.get(cv2.CAP_PROP_FPS))

out = cv2.VideoWriter(
    "../videos/final_trajectory_v2.mp4",
    cv2.VideoWriter_fourcc(*'avc1'),
    fps,
    (width, height)
)

In [60]:
trajectory = []

while True:

    ret, frame = cap.read()

    if not ret:
        break

    results = model(frame, conf=0.1)[0]

    if results.boxes is not None and len(results.boxes) > 0:

        box = results.boxes.xyxy.cpu().numpy()[0]

        x1, y1, x2, y2 = box

        cx = int((x1+x2)/2)
        cy = int((y1+y2)/2)

        
        if len(trajectory) == 0:

            trajectory.append((cx,cy))

        else:

            prev_x, prev_y = trajectory[-1]

            distance = np.sqrt(
                (cx-prev_x)**2 +
                (cy-prev_y)**2
            )

            if distance < 40:
                trajectory.append((cx,cy))

        cv2.rectangle(
            frame,
            (int(x1), int(y1)),
            (int(x2), int(y2)),
            (0,255,0),
            2
        )

        if len(trajectory)>=5:

            smooth_points=[]

            WINDOW=7

            for i in range(len(trajectory)):

                start=max(0,i-WINDOW)

                median_x = int(
                    np.median(
                        [p[0] for p in trajectory[start:i+1]]
                    )
                )

                median_y = int(
                    np.median(
                        [p[1] for p in trajectory[start:i+1]]
                    )
                )

                smooth_points.append((median_x, median_y))

                MAX_TRAIL = 60

                display_points = smooth_points[-MAX_TRAIL:]

                cv2.polylines(
                    frame,
                    [np.array(display_points)],
                    False,
                    (0,0,255),
                    3
                )

    out.write(frame)

cap.release()
out.release()


0: 384x640 2 balls, 45.8ms
Speed: 1.9ms preprocess, 45.8ms inference, 0.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 balls, 35.8ms
Speed: 0.9ms preprocess, 35.8ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 balls, 35.0ms
Speed: 1.2ms preprocess, 35.0ms inference, 0.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 balls, 35.7ms
Speed: 1.4ms preprocess, 35.7ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 balls, 33.1ms
Speed: 1.0ms preprocess, 33.1ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 balls, 33.9ms
Speed: 1.1ms preprocess, 33.9ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 balls, 32.2ms
Speed: 0.8ms preprocess, 32.2ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 balls, 33.9ms
Speed: 0.9ms preprocess, 33.9ms inference, 0.3ms postprocess per image at shape (1, 3, 384, 640)
